In [ ]:
import dspy
from datasets import load_dataset

# 1. Initialisierung der Modelle (lokal betrieben)
# Annahme: Ein Embedding-Modell-Server läuft auf Port 8081
embedder = dspy.Embedder(
    "openai/embeddinggemma-300M-Q8_0.gguf", 
    api_base="http://localhost:8081/v1", 
    api_key="no_key_needed", 
    batch_size=100
)

# Annahme: Ein LLM-Server läuft auf Port 8080
local_llm = dspy.LM(
    "openai/gemma-3-4b-it-Q4_K_M.gguf", 
    api_base="http://localhost:8080/v1", 
    api_key="no_key_needed",
    temperature=0.1,
    cache=False
)

dspy.configure(lm=local_llm, embedder=embedder)

In [ ]:
# Laden des Datensatzes
dataset = load_dataset("embedding-data/simple-wiki", split="train[:1000]")
documents = [" ".join(doc['set']) for doc in dataset] 
# documents[5:]
len (documents)

In [ ]:
# Aus den Dokumenten Chunks erzeugen
chunk_size = 512
all_chunks = []

for doc in documents:
    # Anzahl der Chunks für dieses Dokument
    for i in range(0, len(doc), chunk_size):
        chunk = doc[i:i+chunk_size]
        if len(chunk) > 100:        # nur Chunks > 100 Zeichen übernehmen
            all_chunks.append(chunk)

print(len(all_chunks))
# print(all_chunks[:2])  # die ersten zwei Chunks anzeigen

In [ ]:
# Erstellen der Qdrant Collection (nur falls sie nicht existiert)
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

# Qdrant Client erstellen
client = QdrantClient(host="localhost", port=6333)

collection_name = "simple_wiki_rag"
embedding_dim = 768  # Dimension für embedding-gemma-Modell

# Prüfen, ob die Collection existiert
if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(
            size=embedding_dim,
            distance=Distance.COSINE
        )
    )
    print(f"Collection '{collection_name}' created successfully.")
else:
    print(f"Collection '{collection_name}' already exists.")


In [ ]:
# Embedden und Indexieren der Dokumente
embeddings = embedder(all_chunks)
len(embeddings)

In [ ]:
from qdrant_client.models import PointStruct

points = [
    PointStruct(id=i, vector=vec, payload={"text": chunk})
    for i, (chunk, vec) in enumerate(zip(all_chunks, embeddings))
]

In [ ]:
# Punkte in Qdrant hochladen (in Batches für große Daten)
batch_size = 100
for i in range(0, len(points), batch_size):
    batch = points[i:i+batch_size]
    client.upsert(
        collection_name=collection_name,
        points=batch
    )

print(f"{len(points)} Chunks erfolgreich in Qdrant gespeichert.")

In [ ]:
# 1. Definition des Qdrant Retrievers
class QdrantRetriever(dspy.Retrieve):
    def __init__(self, client, collection_name, embedder, k=3):
        self._client = client
        self._collection_name = collection_name
        self._embedder = embedder
        self._k = k
        super().__init__()

    def forward(self, query_or_queries, k=None):
        k = k if k is not None else self._k
        # Embedden der Suchanfrage
        query_embeddings = self._embedder(query_or_queries)

        # Suche in Qdrant
        results = [
            self._client.query_points(
                collection_name=collection_name,
                query=query_embeddings,
                limit=3,
            ) for emb in query_embeddings
        ]

        # Extrahieren des Textes aus den Suchergebnissen - TODO not only results[0]
        passages = [p.payload["text"] for p in results[0].points]

        return passages

# 2. Definition der Signatur für den Generator
class GenerateAnswer(dspy.Signature):
    """Beantworte die Frage basierend auf dem bereitgestellten Kontext."""
    context = dspy.InputField(desc="Relevante Fakten zur Beantwortung der Frage.")
    question = dspy.InputField(desc="Die ursprüngliche Nutzerfrage.")
    answer = dspy.OutputField(desc="Eine prägnante und faktenbasierte Antwort.")

# 3. Aufbau des RAG-Moduls
class RAG(dspy.Module):
    def __init__(self):
        super().__init__()
        self.retriever = QdrantRetriever(client, collection_name, embedder)
        self.generate_answer = dspy.ChainOfThought(GenerateAnswer)

    def forward(self, question):
        context = self.retriever(question)
        prediction = self.generate_answer(context=context, question=question)
        return dspy.Prediction(context=context, answer=prediction.answer)

In [ ]:
# Instanziieren und Ausführen des RAG-Programms. Zu der Frage gibt es kein passenden Eintrag im Vector Store
rag_pipeline = RAG()
question = "What is the largest continent?"
prediction = rag_pipeline(question)

# Anzeigen der Ergebnisse
print(f"Frage: {question}")
print(f"Antwort: {prediction.answer}")

In [ ]:
local_llm.inspect_history(n=1)

In [ ]:
# Instanziieren und Ausführen des RAG-Programms. Zu der Frage gibt es einen passenden Eintrag im Vector Store
rag_pipeline = RAG()
question = "What happened to the orchestra's funding in 1939, and how did this change the orchestra's governance structure?"
prediction = rag_pipeline(question)

# Anzeigen der Ergebnisse
print(f"Frage: {question}")
print(f"Antwort: {prediction.answer}")

In [ ]:
local_llm.inspect_history(n=1)